# Phase 1 — footage and pretrained-model feasibility

This notebook runs the same intake/pilot modules as the CLI. It does not estimate fatigue or cadence. Install with `uv sync --locked --group dev --group notebook`, then launch Jupyter from this project. SoccerTrack access and the selected video download must be completed first.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/pilot.yaml').exists())
sys.path.insert(0, str(ROOT))
from src.common import load_config
config = load_config(ROOT / 'configs/pilot.yaml')
print('Video:', config['video']['path'])
print('Exists:', config['video']['path'].is_file())

In [ ]:
from src.environment import check_environment
report = check_environment(ROOT / 'results/phase1/notebook_environment.json')
assert report['status'] == 'passed', report['errors']
report['cuda']

In [ ]:
from src.intake import inspect_video
manifest = inspect_video(config)
assert manifest['status'] != 'awaiting_video', manifest.get('action')
manifest['video']

In [ ]:
from IPython.display import display, Image
for path in sorted((config['output_dir'] / 'intake_frames').glob('*.jpg'))[:2]:
    display(Image(filename=str(path), width=1200))

## Run and review the pilot

Detection indices are frame-local and crops contain anonymous people, including possible referees. Complete the manual review fields in `crop_audit.csv`. Confidence is not anatomical accuracy. Source pixels and coordinate-space validation determine whether gait and metric positions are feasible.

In [ ]:
from src.benchmark import run_pilot
pilot = run_pilot(config)
{k: pilot[k] for k in ['sampled_frames', 'crop_audits', 'mean_detector_seconds_per_sample', 'mean_pose_seconds_per_audited_crop', 'peak_torch_vram_mib']}

In [ ]:
import pandas as pd
audit = pd.read_csv(config['output_dir'] / 'crop_audit.csv')
display(audit)
for path in sorted((config['output_dir'] / 'audit_crops').glob('*_pose.jpg'))[:6]:
    display(Image(filename=str(path)))

## Feasibility decision

Record source-pixel detail and manual joint usability, misses/false positives, camera-map coordinate spaces, and the settings chosen for tracking. The proposed study manifest remains provisional until players and early/late availability are verified. Use native-rate continuous running bouts in Phase 3; do not derive cadence from these sparse audit samples.